In [1]:
import requests
import json

def launch_binder_and_get_jupyter_url(binder_url, connect_timeout=10):
    """
    Launch a BinderHub build using requests with streaming enabled.

    Returns:
        (True, jupyter_url_with_token) on success
        (False, error_message) on failure

    Notes:
    - This function blocks until BinderHub reports either success or failure.
    - No exceptions are raised; all errors are converted to return values.
    """

    # Step 1: initiate the HTTP request (may block until headers are received)
    try:
        resp = requests.get(
            binder_url,
            stream=True,
            timeout=(connect_timeout, None),  # timeout only for connection, not for streaming
        )
    except Exception as e:
        return False, f"Failed to connect to BinderHub: {e}"

    # Step 2: validate HTTP response
    if resp.status_code != 200:
        return False, f"BinderHub returned HTTP {resp.status_code}"

    # Step 3: read Server-Sent Events (SSE) line by line
    try:
        for raw_line in resp.iter_lines(decode_unicode=True):
            if not raw_line:
                continue

            # BinderHub sends events in SSE format: "data: {...}"
            if not raw_line.startswith("data:"):
                continue

            data = raw_line[5:].strip()

            # Parse JSON payload
            try:
                event = json.loads(data)
            except json.JSONDecodeError:
                continue

            phase = event.get("phase")

            # ---- Optional: real-time logging ----
            # print("EVENT:", event, flush=True)

            # Build failed
            if phase == "failed":
                message = event.get("message") or "Binder build failed"
                return False, message

            # Build succeeded and Jupyter server is ready
            if phase == "ready":
                url = event.get("url")
                token = event.get("token")

                if not url:
                    return False, "Binder reported ready phase without a URL"

                # Some BinderHub versions include the token separately,
                # others already embed it in the URL
                # if token and "token=" not in url:
                #     separator = "&" if "?" in url else "?"
                #     url = f"{url}{separator}token={token}"

                return True, {"url": url, "token": token}

        # Stream ended without receiving a final "ready" or "failed" event
        return False, "Binder event stream ended without a final result"

    except Exception as e:
        return False, f"Error while reading Binder event stream: {e}"


In [ ]:
binder_base_url = "https://binder.intel4coro.de/build/gh"
repo_path = "yxzhan/cram_isaacsim/main"

launch_url = f"{binder_base_url}/{repo_path}"

ok, result = launch_binder_and_get_jupyter_url(launch_url)

if ok:
    print("JupyterLab URL:", result)
else:
    print("Binder failed:", result)

JupyterLab URL: {'url': 'https://jupyter.intel4coro.de/user/yxzhan-fr3-genesis-eec0yd3e/', 'token': 'QiMpQaMRRIiqHBJ8hlBEnA'}


In [3]:
baseurl = result["url"]
token = result["token"]

In [ ]:
session = requests.Session()
session.get(f"{baseurl}?token={token}")

<Response [200]>

In [ ]:
print("Status code:", resp.status_code)
for k, v in resp.headers.items():
    print(f"{k}: {v}")

print(resp.text)